# 02 — Duration Regression

This notebook models trip duration as a regression problem using only pre-trip features.

The production framing: before a rider leaves, estimate how long the Manhattan → JFK/LGA ride is likely to take.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

DATA = Path('../data/processed/taxi_clean_for_modeling.csv')
df = pd.read_csv(DATA, parse_dates=['tpep_pickup_datetime', 'tpep_dropoff_datetime'])
if 'duration_min' not in df.columns:
    df['duration_min'] = df['trip_duration_min']

df = df.sort_values('tpep_pickup_datetime').reset_index(drop=True)
df.shape

## Pre-Trip Features Only

The model excludes fare, toll, tip, and total amount because those are known after the ride and would leak information about trip duration.


In [ ]:
features = ['trip_distance', 'passenger_count', 'pickup_hour', 'pickup_dow', 'payment_type', 'airport', 'PULocationID']
target = 'duration_min'

X = df[features].copy()
y = df[target].copy()

split_idx = int(len(df) * 0.80)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

X_train.shape, X_test.shape

## Baseline: Historical Median

A strong baseline predicts the training median duration for every trip. The full project also tested more granular median baselines by pickup zone, airport, and hour.


In [ ]:
baseline_pred = np.full_like(y_test, y_train.median(), dtype=float)

def reg_metrics(y_true, y_pred):
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'R2': r2_score(y_true, y_pred),
    }

reg_metrics(y_test, baseline_pred)

## Random Forest Model

Tree-based models fit this problem well because traffic patterns are nonlinear and interact with route, time, and pickup zone.


In [ ]:
cat_cols = ['airport', 'PULocationID', 'payment_type']
num_cols = ['trip_distance', 'passenger_count', 'pickup_hour', 'pickup_dow']

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
    ('num', 'passthrough', num_cols),
])

rf = Pipeline([
    ('preprocess', preprocessor),
    ('model', RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1
    ))
])

rf.fit(X_train, y_train)
preds = rf.predict(X_test)
reg_metrics(y_test, preds)

## Portfolio Result Summary

In the full tuned notebook, Random Forest and XGBoost both produced roughly **5.6 minute MAE**, cutting error by more than half versus the median baseline.
